# Phase 7: Resolve GBIF taxon references

Match each Phase 5 coordinate species against the GBIF taxonomic Backbone and save the association table at `sites/pathogen-atlas-explorer/data/species_gbif_links.csv`. The species domain is supplied as a taxonomy hint. A GBIF link is recorded only for an exact canonical-name match at species rank; fuzzy or higher-rank matches are retained as diagnostics but do not produce a link.

Re-running this notebook reuses completed lookup rows. Set `FORCE_REFRESH = True` only when you want to repeat all GBIF API requests.

The [GBIF API beginner guide](https://data-blog.gbif.org/post/gbif-api-beginners-guide/) demonstrates the v1 name-matching endpoint and its `usageKey`. This resolver uses the current [GBIF species API](https://techdocs.gbif.org/en/openapi/v1/species) batch endpoint so the 1,703 names can be handled with a small number of requests; its equivalent taxon key is `usage.key`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "pathogen_atlas_v2").is_dir():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Open this notebook from the Pathogen Atlas version2 repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent

COORDINATES_PATH = PROJECT_ROOT / "data/05_plots/release/species_feature_space_coordinates.parquet"
SPECIES_SUMMARY_PATH = PROJECT_ROOT / "data/04_tables/release/species_summary.parquet"
LOOKUP_TABLE_PATH = PROJECT_ROOT / "sites/pathogen-atlas-explorer/data/species_gbif_links.csv"
BUILDER_SCRIPT = PROJECT_ROOT / "sites/pathogen-atlas-explorer/scripts/build_gbif_links.py"
FORCE_REFRESH = False

for required_path in (COORDINATES_PATH, SPECIES_SUMMARY_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Run Phases 4 and 5 first; required input is missing: {required_path}")

species_count = pd.read_parquet(COORDINATES_PATH, columns=["canonical_species"])["canonical_species"].nunique()
existing_count = 0
if LOOKUP_TABLE_PATH.exists():
    existing_count = len(pd.read_csv(LOOKUP_TABLE_PATH, keep_default_na=False))
display(Markdown(
    f"**Lookup scope:** {species_count:,} coordinate species  ·  "
    f"**Existing associations:** {existing_count:,} rows  ·  "
    f"**Force refresh:** {FORCE_REFRESH}"
))

## Resolve and save the GBIF association table

The GBIF v2 batch-matching endpoint resolves up to 100 names per request. Completed batches are written immediately, so interrupted runs resume from the saved rows. Only exact canonical species matches receive a taxon page URL.

In [ ]:
command = [sys.executable, str(BUILDER_SCRIPT)]
if FORCE_REFRESH:
    command.append("--force-refresh")
subprocess.run(command, cwd=PROJECT_ROOT, check=True)

In [ ]:
gbif_links = pd.read_csv(LOOKUP_TABLE_PATH, keep_default_na=False)
gbif_found = gbif_links["match_status"].eq("found")
display(Markdown(
    f"**Saved:** `{LOOKUP_TABLE_PATH.relative_to(PROJECT_ROOT)}`  ·  "
    f"**Species:** {len(gbif_links):,}  ·  "
    f"**Exact GBIF species pages:** {int(gbif_found.sum()):,}  ·  "
    f"**No exact species-page match:** {int((~gbif_found).sum()):,}"
))
display(gbif_links.head(12))